In [14]:
import warnings 
warnings.filterwarnings("ignore") 
import numpy as np 
import pandas as pd 
import dask 
import dask.dataframe as dd
import os
  
def clean_partition(pdf): 
    """Same standardisation rules as Question 2, already written in pandas for you.""" 
    pdf = pdf.copy() 
    pdf["Product_Category"] = pdf["Product_Category"].str.strip().str.title() 
    pdf["Order_Status"] = pdf["Order_Status"].str.strip().str.title() 
    key = pdf["Payment_Method"].str.lower().str.replace(r"[^a-z]", "", regex=True) 
    pay_map = {"creditcard": "Credit Card", "debitcard": "Debit Card", "paypal": "PayPal", 
               "banktransfer": "Bank Transfer", "applepay": "Apple Pay", "cash": "Cash"} 
    pdf["Payment_Method"] = key.map(pay_map).fillna("Unknown") 
    # one timestamp column; a sub-second tie-breaker from Transaction_ID keeps the sort order deterministic 
    ts = pd.to_datetime(pdf["Order_Date"] + " " + pdf["Order_Time"], errors="coerce") 
    tie = pd.to_numeric(pdf["Transaction_ID"].str[1:], errors="coerce").fillna(0).astype("int64") % 1_000_000 
    pdf["Order_Timestamp"] = ts + pd.to_timedelta(tie, unit="us") 
    return pdf 
  
ddf = (dd.read_csv("./Sales_transactions_2022_2025.csv") 
         .drop_duplicates()                    # exact duplicate rows removed (18,045 -> 18,000) 
         .map_partitions(clean_partition) 
         .repartition(npartitions=5))          # 5 partitions for the whole Dask part 
  
print("Partitions:", ddf.npartitions)

Partitions: 5


In [15]:
print("Total Partitions:", ddf.npartitions)
partition_sizes = ddf.map_partitions(len).compute()
print("Partition Lengths:", partition_sizes.tolist())
print("Total Row Count:", partition_sizes.sum())

Total Partitions: 5
Partition Lengths: [3600, 3600, 3600, 3600, 3600]
Total Row Count: 18000


In [16]:
# (i) Groupby Product_Category loss rate calculation
def calc_partition_loss(df_part, group_col):
    # Count total and loss rows per category in each partition
    grouped = df_part.groupby(group_col).agg(
        loss_count=('Profit', lambda x: (x < 0).sum()),
        total_count=('Profit', 'count')
    )
    return grouped

# Compute partial aggregations across partitions and sum them up globally
cat_agg = ddf.map_partitions(
    calc_partition_loss, 
    "Product_Category", 
    meta={'loss_count': 'int64', 'total_count': 'int64'}
).compute().groupby(level=0).sum()

cat_loss = (cat_agg['loss_count'] / cat_agg['total_count']) * 100
cat_loss = cat_loss.sort_values(ascending=False)
print("--- Product Category Loss Rates ---")
print(cat_loss)


# (ii) Groupby Discount_Percentage loss rate calculation
disc_agg = ddf.map_partitions(
    calc_partition_loss, 
    "Discount_Percentage", 
    meta={'loss_count': 'int64', 'total_count': 'int64'}
).compute().groupby(level=0).sum()

disc_loss = (disc_agg['loss_count'] / disc_agg['total_count']) * 100
print("\n--- Discount Levels with >10% Loss Rate ---")
print(disc_loss[disc_loss > 10])

--- Product Category Loss Rates ---
Product_Category
Office Supplies    2.873563
Electronics        0.915975
Furniture          0.544959
Appliances         0.403023
dtype: float64

--- Discount Levels with >10% Loss Rate ---
Discount_Percentage
20.0    10.156971
25.0    29.874214
30.0    60.000000
dtype: float64


In [17]:
mem_before = ddf.memory_usage(deep=True).compute().sum() / (1024 ** 2)

cat_cols = ["Product_Category", "Sales_Channel", "Customer_Segment", 
            "Shipping_Method", "Payment_Method", "Order_Status", "Country"]

ddf_cat = ddf.categorize(columns=cat_cols)
mem_after = ddf_cat.memory_usage(deep=True).compute().sum() / (1024 ** 2)

pct_reduction = ((mem_before - mem_after) / mem_before) * 100
print(f"Memory Usage Before Categorization: {mem_before:.2f} MB")
print(f"Memory Usage After Categorization:  {mem_after:.2f} MB")
print(f"Memory Reduction:                   {pct_reduction:.2f}%")

Memory Usage Before Categorization: 8.84 MB
Memory Usage After Categorization:  6.84 MB
Memory Reduction:                   22.62%


In [18]:
ddf_cat.to_parquet("sales_by_country.parquet", partition_on=["Country"], write_index=False)

# Inspect subdirectories
print("Country folders:", [d for d in os.listdir("sales_by_country.parquet") if d.startswith("Country=")])

# Read back using predicate pushdown filter
canada_df = dd.read_parquet("sales_by_country.parquet", filters=[("Country", "==", "Canada")]).compute()
direct_canada_count = (ddf["Country"] == "Canada").sum().compute()

print(f"Canada rows read from Parquet filter: {len(canada_df)}")
print(f"Canada rows computed directly from ddf: {direct_canada_count}")

Country folders: ['Country=Australia', 'Country=Canada', 'Country=France', 'Country=Germany', 'Country=United Kingdom', 'Country=United States']
Canada rows read from Parquet filter: 2680
Canada rows computed directly from ddf: 2680


In [19]:
ddf_sorted = ddf.set_index("Order_Timestamp")
print("Partition count after set_index:", ddf_sorted.npartitions)
print("First Division Boundary:", ddf_sorted.divisions[0])
print("Last Division Boundary: ", ddf_sorted.divisions[-1])

Partition count after set_index: 5
First Division Boundary: 2022-01-01 14:32:19.010888
Last Division Boundary:  2025-12-31 19:03:43.003711


In [20]:
roll_50 = ddf_sorted["Sales_Amount"].rolling(window=50).mean()
max_val = roll_50.max().compute()
max_idx = roll_50.idxmax().compute()

print(f"Maximum 50-Txn Rolling Mean: ${max_val:.2f} at {max_idx}")

# Boundary Inspection across Partition 1 boundary
div1 = ddf_sorted.divisions[1]
boundary_sample = roll_50.loc[div1 - pd.Timedelta(seconds=1): div1 + pd.Timedelta(seconds=1)].compute()
print("\nRolling values straddling division boundary:\n", boundary_sample.head(5))

Maximum 50-Txn Rolling Mean: $1062.64 at 2022-10-24 13:20:02.007985

Rolling values straddling division boundary:
 Order_Timestamp
2022-11-16 14:16:48.002623    416.4604
Name: Sales_Amount, dtype: float64


In [22]:
# 1. Compute rolling mean of the previous 20 transactions across the sorted Dask Series
# rolling(20, min_periods=20).mean().shift(1) works natively on ddf_sorted['Sales_Amount']
prev20_mean = ddf_sorted["Sales_Amount"].rolling(window=20, min_periods=20).mean().shift(1)

# 2. Define the spike condition: Sales_Amount > 3 * prev20_mean
is_spike = ddf_sorted["Sales_Amount"] > (3 * prev20_mean)

# 3. Combine into a result DataFrame
spikes_df = ddf_sorted[['Sales_Amount']].assign(
    prev20_mean=prev20_mean,
    is_spike=is_spike
)

# 4. Filter for spikes and compute the results
spikes_computed = spikes_df[spikes_df['is_spike']].compute()

print("Number of spikes found:", len(spikes_computed))
print("\n5 largest spikes by Sales_Amount:")
print(spikes_computed.nlargest(5, 'Sales_Amount'))

Number of spikes found: 1572

5 largest spikes by Sales_Amount:
                            Sales_Amount  prev20_mean  is_spike
Order_Timestamp                                                
2024-08-29 11:15:48.015568      14303.07     685.8360      True
2023-10-25 12:42:05.000197      13451.72     588.2945      True
2023-04-26 13:29:06.010546      12887.91     462.3875      True
2022-08-22 18:06:46.008243      12805.91     488.4835      True
2024-06-26 08:37:11.006286      12491.61     487.0840      True


In [23]:
ddf_spikes_mp = ddf_sorted.map_partitions(flag_spikes, meta=meta_struct)

res_overlap = ddf_spikes.compute()
res_partitions = ddf_spikes_mp.compute()

print("map_overlap Spikes Count:   ", res_overlap["is_spike"].sum())
print("map_partitions Spikes Count:", res_partitions["is_spike"].sum())

print("map_overlap missing prev20_mean count:   ", res_overlap["prev20_mean"].isna().sum())
print("map_partitions missing prev20_mean count:", res_partitions["prev20_mean"].isna().sum())

map_overlap Spikes Count:    1572
map_partitions Spikes Count: 1564
map_overlap missing prev20_mean count:    20
map_partitions missing prev20_mean count: 100


In [26]:
# 1. Fill missing values with column means
means = ddf[["Customer_Age", "Delivery_Days", "Customer_Rating"]].mean().compute()
ddf_ml = ddf.fillna(means.to_dict())

# 2. Create target variable safely as float/int (0.0 or 1.0)
ddf_ml["is_loss"] = (ddf_ml["Profit"] < 0).astype(float)

# 3. Drop leak columns, target, and all non-feature columns
features_df = ddf_ml.drop(columns=[
    "Profit", "Sales_Amount", "Cost_Amount", "Transaction_ID", "Order_ID", 
    "Customer_ID", "Customer_Name", "Order_Date", "Order_Time", "Store_ID", 
    "Store_Name", "Region", "City", "Product_ID", "Product_Name", 
    "Product_Subcategory", "Return_Flag", "Return_Reason", "Sales_Representative", 
    "Promotion_Code", "Inventory_Level", "Order_Year", "Order_Timestamp"
])

# Separate X and y clearly
X = features_df.drop(columns=["is_loss"])
y = features_df["is_loss"]

# 4. Explicitly categorize the 5 categorical columns
cat_cols = ["Product_Category", "Sales_Channel", "Customer_Segment", "Shipping_Method", "Payment_Method"]
X = X.categorize(columns=cat_cols)

# 5. One-hot encode features
catz = Categorizer()
X_cat = catz.fit_transform(X)
dummy = DummyEncoder()
X_encoded = dummy.fit_transform(X_cat).astype("float64")

# 6. Train/Test Split and Scaling
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.25, shuffle=True, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Loss rate:", y.mean().compute() * 100)
print("Number of feature columns:", X_encoded.shape[1])
print("Train / Test rows:", len(y_train), len(y_test))

Loss rate: 1.2833333333333334
Number of feature columns: 48
Train / Test rows: 13457 4543


In [30]:
from dask_ml.wrappers import Incremental
from sklearn.linear_model import SGDClassifier

# 1. Initialize SGDClassifier without the classes argument
sgd = SGDClassifier(
    loss="log_loss", 
    penalty="l2", 
    random_state=42
)

# 2. Wrap it with Incremental
model = Incremental(sgd)

# 3. Convert training sets to Dask arrays
X_train_arr = X_train_scaled.to_dask_array(lengths=True)
y_train_arr = y_train.to_dask_array(lengths=True)

# 4. Pass classes=[0, 1] directly to model.fit()
print("Training array partitions:", X_train_arr.npartitions)
model.fit(X_train_arr, y_train_arr, classes=[0, 1])

print("Model successfully fitted!")

Training array partitions: 5
Model successfully fitted!


In [32]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Compute Dask arrays to numpy arrays for sklearn evaluation compatibility
X_train_np = X_train_scaled.compute() if hasattr(X_train_scaled, "compute") else X_train_scaled
y_train_np = y_train.compute() if hasattr(y_train, "compute") else y_train
X_test_np = X_test_scaled.compute() if hasattr(X_test_scaled, "compute") else X_test_scaled
y_test_np = y_test.compute() if hasattr(y_test, "compute") else y_test

# 2. Define and fit the baseline model (e.g., predicting the most frequent class)
model_baseline = DummyClassifier(strategy="most_frequent", random_state=42)
model_baseline.fit(X_train_np, y_train_np)

# 3. Evaluate the baseline model
baseline_preds = model_baseline.predict(X_test_np)
baseline_acc = accuracy_score(y_test_np, baseline_preds)

print(f"Baseline Accuracy: {baseline_acc:.4f}")

# 4. Evaluate your trained Incremental model (`model`) for comparison
model_preds = model.predict(X_test_scaled)
model_acc = accuracy_score(y_test_np, model_preds.compute() if hasattr(model_preds, "compute") else model_preds)

print(f"Incremental Model Accuracy: {model_acc:.4f}")

Baseline Accuracy: 0.9857
Incremental Model Accuracy: 0.9859
